# V7_A_N08 — Rainfall and Drought Impact Monitoring

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft using synthetic data. Outputs support authorized human review; they do not constitute official declarations or automated decisions.

## Decision contract
Prioritize field verification and preparedness during the crop season. Owners: meteorological, agriculture, disaster-management, and statistical authorities. No indicator alone declares drought, crop failure, or beneficiary eligibility.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7808); stations=[f'S{i:02d}' for i in range(12)]; dekads=np.arange(1,19); rows=[]
for s in stations:
 normal=rng.uniform(28,48)
 for d in dekads: rows.append((s,d,normal,max(0,rng.normal(normal*(.55 if d in [7,8,9] else 1),8)),rng.uniform(.65,1)))
df=pd.DataFrame(rows,columns=['station','dekad','normal_mm','rain_mm','satellite_quality']); df.head()

## Quality and coverage
Station completeness, calibration, location, and reference climatology matter. Satellite signals complement stations and field observation; they are not automatic ground truth.

In [2]:
df['completeness']=rng.uniform(.72,1,len(df)); df['usable']=(df.completeness>=.85)&(df.satellite_quality>=.75); print(df.usable.mean(),df.groupby('station').usable.mean().round(2).to_string())

0.32407407407407407 station
S00    0.33
S01    0.28
S02    0.39
S03    0.28
S04    0.22
S05    0.33
S06    0.39
S07    0.28
S08    0.56
S09    0.22
S10    0.22
S11    0.39


## Anomaly and cumulative deficit
A rainfall anomaly is relative to an explicit normal. A deficit does not equal crop loss; impact depends on crop, growth stage, soils, irrigation, and management.

In [3]:
df['anomaly_pct']=100*(df.rain_mm-df.normal_mm)/df.normal_mm; df['deficit_mm']=(df.normal_mm-df.rain_mm).clip(lower=0); cum=df.groupby('station').agg(rain=('rain_mm','sum'),normal=('normal_mm','sum'),usable=('usable','mean')); cum['season_deficit_pct']=100*(cum.normal-cum.rain)/cum.normal; print(cum.round(1).to_string())

          rain  normal  usable  season_deficit_pct
station                                           
S00      757.5   842.1     0.3                10.0
S01      699.9   683.0     0.3                -2.5
S02      743.1   821.8     0.4                 9.6
S03      634.6   760.2     0.3                16.5
S04      715.3   756.5     0.2                 5.5
S05      690.5   815.1     0.3                15.3
S06      532.1   602.0     0.4                11.6
S07      554.9   550.7     0.3                -0.8
S08      579.3   558.7     0.6                -3.7
S09      769.3   839.9     0.2                 8.4
S10      655.3   678.1     0.2                 3.4
S11      677.7   704.0     0.4                 3.7


## Crop-calendar weighting
The same rainfall deficit can have different consequences by growth stage. Calendar weights are agronomic assumptions requiring local validation.

In [4]:
stage=np.select([df.dekad<=5,df.dekad<=10,df.dekad<=14],['establishment','flowering','grain_fill'],default='maturity'); df['stage']=stage; weights={'establishment':.8,'flowering':1.4,'grain_fill':1.1,'maturity':.4}; df['impact_signal']=df.deficit_mm*df.stage.map(weights)*df.usable; risk=df.groupby('station').impact_signal.sum().sort_values(ascending=False); print(risk.head().round(1).to_string())

station
S09    86.0
S05    70.8
S00    63.8
S06    55.1
S03    52.0


## Sensitivity and abstention
Compare rankings under alternative stage weights. Stations with inadequate usable coverage are routed to data review.

In [5]:
alt={'establishment':1.0,'flowering':1.1,'grain_fill':1.3,'maturity':.4}; alt_risk=(df.deficit_mm*df.stage.map(alt)*df.usable).groupby(df.station).sum(); overlap=len(set(risk.head(5).index)&set(alt_risk.nlargest(5).index))/5; out=cum.assign(impact_signal=risk,decision=np.where(cum.usable<.8,'ABSTAIN—DATA REVIEW','FIELD VERIFICATION')); print('TOP5_OVERLAP',overlap); print(out.sort_values('impact_signal',ascending=False).head().round(2).to_string())

TOP5_OVERLAP 1.0
           rain  normal  usable  season_deficit_pct  impact_signal             decision
station                                                                                
S09      769.33  839.92    0.22                8.40          85.98  ABSTAIN—DATA REVIEW
S05      690.50  815.12    0.33               15.29          70.80  ABSTAIN—DATA REVIEW
S00      757.52  842.11    0.33               10.05          63.82  ABSTAIN—DATA REVIEW
S06      532.09  602.02    0.39               11.62          55.12  ABSTAIN—DATA REVIEW
S03      634.60  760.20    0.28               16.52          51.99  ABSTAIN—DATA REVIEW


## Exercises
1. Add a standardized precipitation index with a documented reference period. 2. Compare station-only and satellite-assisted results. 3. Add crop-specific calendars. 4. Explain why a drought alert cannot determine compensation.

## Exact solutions
1. Fit/standardize precipitation over an approved long reference series and report distribution assumptions. 2. Evaluate bias, coverage, spatial representativeness, and disagreement; route large disagreement to verification. 3. Join locally validated crop-stage dates and propagate calendar uncertainty. 4. Compensation also requires legal criteria, verified losses, exposure, eligibility, budgets, appeals, and authorized decisions.

In [6]:
assert overlap>=0 and out.decision.isin(['ABSTAIN—DATA REVIEW','FIELD VERIFICATION']).all(); print('V7_A_N08_REWORK_COMPLETE_EXECUTION_PASS')

V7_A_N08_REWORK_COMPLETE_EXECUTION_PASS
